# Telco Customer Churn — End-to-End Exploratory Data Analysis

**Objective**

To analyze customer churn behavior for a telecommunications provider using historical customer account data. To compare churn rates across contract types, service types, and demographic groups. To identify churn trends and patterns using exploratory data analysis and statistical analysis. To evaluate variations in churn based on tenure, billing method, and subscribed services. To generate data-driven insights and recommendations through Python-based data visualization and analytics.

**Dataset Information**

Source: IBM Sample Data Sets — Telco Customer Churn

Location: United States (California)

Year/Timeline: Cross-sectional snapshot of active and churned customer accounts (no explicit date range in the source data)

Domain: Telecommunications / Customer Retention

**Business Problem**

Customer churn — subscribers discontinuing service — is one of the most significant challenges facing telecommunications companies, directly impacting revenue and long-term growth.

Retention teams need accurate, data-driven analysis to understand which customer segments (by contract type, tenure, service bundle, payment method, and demographics) are most likely to churn.

However, the volume and variety of account, billing, and service data makes it difficult to identify meaningful patterns without proper analysis.

This project aims to analyze the Telco Customer Churn dataset using Python to generate insights that support proactive, targeted retention strategies.

**Attribute Table**

| Attribute Name | Data Type | Description |
|---|---|---|
| CustomerID | String (Object) | Unique customer identifier |
| City / Zip Code / Lat Long | String / Float | Customer's service location |
| Gender | String (Object) | Customer's gender |
| Senior Citizen | String (Object) | Whether the customer is 65+ |
| Partner / Dependents | String (Object) | Household composition |
| Tenure Months | Integer | Number of months as a customer |
| Phone Service / Internet Service | String (Object) | Core services subscribed |
| Online Security, Online Backup, Device Protection, Tech Support, Streaming TV, Streaming Movies | String (Object) | Add-on services subscribed |
| Contract | String (Object) | Month-to-month, One year, Two year |
| Paperless Billing / Payment Method | String (Object) | Billing preferences |
| Monthly Charges / Total Charges | Float | Billing amounts |
| Churn Label / Churn Value | String / Integer | Whether the customer churned (Yes/1, No/0) |
| Churn Score | Integer | Propensity-to-churn score (0–100) |
| CLTV | Integer | Customer Lifetime Value |
| Churn Reason | String (Object) | Self-reported reason for churning (churned customers only) |

**Notebook Structure**
1. Data Loading and Initial Overview
2. Data Pre-processing
3. Exploratory Data Analysis (EDA)
4. Visualizations
5. Insight Generation and Summary


## 0. Setup — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display & plotting settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
COLOR_PALETTE = ['#2E86AB', '#E63946', '#F4A261', '#2A9D8F', '#8D99AE']
sns.set_palette(COLOR_PALETTE)

print("Libraries loaded successfully.")

## 1. Data Loading and Initial Overview

We load the dataset directly from the CSV file. If you're running this in **Google Colab**, upload `Telco_Customer_Churn_Dataset.csv` to the Colab file browser (or mount Google Drive) and adjust the path below if needed.

In [ ]:
# If using Google Colab, uncomment the lines below to upload the file interactively
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv('Telco_Customer_Churn_Dataset.csv')
print(f"Dataset loaded: {df.shape[0]} rows and {df.shape[1]} columns")

### 1.1 Shape and Column Overview

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\nColumn names:")
print(list(df.columns))

### 1.2 Data Types

In [ ]:
df.dtypes

### 1.3 First Look at the Data

In [ ]:
df.head()

### 1.4 Structural Overview

In [ ]:
df.info()

### 1.5 Statistical Summary

In [ ]:
# Numerical summary
df.describe()

In [ ]:
# Categorical summary
df.describe(include='object')

**Initial Observations:**
- The dataset has 7,043 customer records and 33 columns covering demographics, subscribed services, billing, and churn outcome.
- `Total Charges` is currently stored as text (`object`) instead of a numeric type — this needs correction.
- `Churn Label` / `Churn Value` indicate whether a customer left (Yes/1) or stayed (No/0); `Churn Reason` is only populated for customers who churned.
- Several service columns (`Online Security`, `Tech Support`, etc.) are categorical with Yes/No/No internet service values.

## 2. Data Pre-processing

### 2.1 Handling Missing Values

In [ ]:
# Check missing values across all columns
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Columns with missing values:")
print(missing)
print(f"\n'Churn Reason' is missing for {df['Churn Reason'].isnull().sum()} rows — this is expected, since only churned customers have a reason recorded.")

In [ ]:
# 'Churn Reason' missing simply means "customer did not churn" — fill with a meaningful label instead of dropping
df['Churn Reason'] = df['Churn Reason'].fillna('Not Churned')

# Confirm no more unexplained missing values
df.isnull().sum().sum()

### 2.2 Correcting Data Types

In [ ]:
# 'Total Charges' should be numeric, but contains blank strings for customers with 0 tenure (new customers)
blank_mask = df['Total Charges'].astype(str).str.strip() == ''
print(f"Rows with blank Total Charges: {blank_mask.sum()}")
df[blank_mask][['CustomerID', 'Tenure Months', 'Total Charges']]

In [ ]:
# These are all customers with 0 months of tenure (haven't been billed yet) — it's logical to set their Total Charges to 0
df['Total Charges'] = df['Total Charges'].replace(' ', np.nan)
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Total Charges'] = df['Total Charges'].fillna(0)

print("Total Charges dtype now:", df['Total Charges'].dtype)
print("Remaining nulls:", df['Total Charges'].isnull().sum())

In [ ]:
# Convert Yes/No flag columns to consistent categorical/boolean-friendly text and ensure numeric columns are correct type
df['Senior Citizen'] = df['Senior Citizen'].astype(str)
df['Zip Code'] = df['Zip Code'].astype(str)  # Zip code is an identifier, not a quantity

df.dtypes[['Total Charges', 'Senior Citizen', 'Zip Code']]

### 2.3 Removing Duplicates

In [ ]:
dup_count = df.duplicated().sum()
print(f"Fully duplicated rows: {dup_count}")

dup_id_count = df['CustomerID'].duplicated().sum()
print(f"Duplicate CustomerIDs: {dup_id_count}")

df = df.drop_duplicates()
print(f"Shape after de-duplication: {df.shape}")

### 2.4 Dropping Redundant / Low-Value Columns

In [ ]:
# 'Count' is always 1 (row counter) and 'Country'/'State' are constant (all customers are in California, USA) —
# they add no analytical value. 'Churn Value' duplicates 'Churn Label' as 0/1, which we'll keep for calculations.
print(df['Count'].unique(), df['Country'].unique(), df['State'].unique())

df = df.drop(columns=['Count', 'Country', 'State'])
df.shape

### 2.5 Creating Derived Columns

In [ ]:
# Tenure Group: bucket tenure (months) into readable ranges for easier segmentation
def tenure_group(months):
    if months <= 12:
        return '0-1 Year'
    elif months <= 24:
        return '1-2 Years'
    elif months <= 48:
        return '2-4 Years'
    elif months <= 60:
        return '4-5 Years'
    else:
        return '5+ Years'

df['Tenure Group'] = df['Tenure Months'].apply(tenure_group)

# Count of subscribed add-on services (Online Security, Backup, Device Protection, Tech Support, Streaming TV/Movies)
service_cols = ['Online Security', 'Online Backup', 'Device Protection',
                 'Tech Support', 'Streaming TV', 'Streaming Movies']
df['Services Subscribed'] = (df[service_cols] == 'Yes').sum(axis=1)

# Average revenue per month of tenure (sanity-checks Monthly Charges against actual billing history)
df['Avg Monthly Spend'] = np.where(df['Tenure Months'] > 0,
                                    df['Total Charges'] / df['Tenure Months'],
                                    df['Monthly Charges'])

# CLTV segment (Customer Lifetime Value), split into tiers for grouped analysis
df['CLTV Segment'] = pd.qcut(df['CLTV'], q=3, labels=['Low', 'Medium', 'High'])

df[['Tenure Months', 'Tenure Group', 'Services Subscribed', 'Avg Monthly Spend', 'CLTV', 'CLTV Segment']].head()

### 2.6 Filtering & Aggregating — Quick Checks

In [ ]:
# Example filter: high-value customers (top CLTV tier) who churned — a high-priority segment to understand
high_value_churned = df[(df['CLTV Segment'] == 'High') & (df['Churn Label'] == 'Yes')]
print(f"High-CLTV customers who churned: {len(high_value_churned)} "
      f"({len(high_value_churned)/len(df[df['CLTV Segment']=='High'])*100:.1f}% of all high-CLTV customers)")

# Example aggregation: average monthly charges and churn rate by contract type
agg_by_contract = df.groupby('Contract').agg(
    customers=('CustomerID', 'count'),
    avg_monthly_charge=('Monthly Charges', 'mean'),
    churn_rate=('Churn Value', 'mean')
).round(2)
agg_by_contract

## 3. Exploratory Data Analysis (EDA)

We now explore the cleaned dataset through univariate, bivariate, and multivariate lenses to uncover patterns related to churn.

### 3.1 Univariate Analysis

In [ ]:
# Overall churn rate
churn_counts = df['Churn Label'].value_counts()
churn_rate = df['Churn Value'].mean() * 100
print(churn_counts)
print(f"\nOverall churn rate: {churn_rate:.2f}%")

In [ ]:
# Central tendency and spread of key numeric variables
df[['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV', 'Churn Score']].describe().T

### 3.2 Bivariate Analysis — Churn vs. Key Categorical Features

In [ ]:
for col in ['Contract', 'Internet Service', 'Payment Method', 'Senior Citizen', 'Paperless Billing']:
    print(f"\n--- Churn rate by {col} ---")
    print(df.groupby(col)['Churn Value'].mean().mul(100).round(2).sort_values(ascending=False))

### 3.3 Multivariate Analysis — GroupBy, Pivot Tables & Correlation

In [ ]:
# Pivot table: average churn rate by Contract type and Internet Service
pivot = pd.pivot_table(df, values='Churn Value', index='Contract',
                        columns='Internet Service', aggfunc='mean').round(3) * 100
pivot

In [ ]:
# Groupby: churn rate and average spend by Tenure Group
tenure_summary = df.groupby('Tenure Group').agg(
    customers=('CustomerID', 'count'),
    churn_rate_pct=('Churn Value', lambda x: round(x.mean()*100, 2)),
    avg_monthly_charges=('Monthly Charges', 'mean'),
    avg_cltv=('CLTV', 'mean')
).reindex(['0-1 Year', '1-2 Years', '2-4 Years', '4-5 Years', '5+ Years'])
tenure_summary

In [ ]:
# Correlation matrix of numeric variables
numeric_cols = ['Tenure Months', 'Monthly Charges', 'Total Charges', 'Churn Score',
                 'CLTV', 'Services Subscribed', 'Avg Monthly Spend', 'Churn Value']
corr_matrix = df[numeric_cols].corr()
corr_matrix

In [ ]:
# Top reasons customers give for churning (excluding those who didn't churn)
reason_counts = df[df['Churn Label'] == 'Yes']['Churn Reason'].value_counts().head(10)
reason_counts

## 4. Visualizations

### 4.1 Churn Distribution (Pie Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
churn_counts.plot(kind='pie', autopct='%1.1f%%', startangle=90,
                   colors=['#2A9D8F', '#E63946'], labels=['No Churn', 'Churn'],
                   ax=ax, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax.set_ylabel('')
ax.set_title('Customer Churn Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2 Churn Rate by Contract Type (Bar Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
contract_churn = df.groupby('Contract')['Churn Value'].mean().mul(100).sort_values(ascending=False)
bars = ax.bar(contract_churn.index, contract_churn.values, color=COLOR_PALETTE[:3], edgecolor='black')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_xlabel('Contract Type')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Tenure Distribution (Histogram)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df[df['Churn Label']=='No']['Tenure Months'], bins=30, alpha=0.6, label='No Churn', color='#2A9D8F')
ax.hist(df[df['Churn Label']=='Yes']['Tenure Months'], bins=30, alpha=0.6, label='Churn', color='#E63946')
ax.set_xlabel('Tenure (Months)')
ax.set_ylabel('Number of Customers')
ax.set_title('Distribution of Customer Tenure by Churn Status', fontsize=14, fontweight='bold')
ax.legend(title='Churn Status')
plt.tight_layout()
plt.show()

### 4.4 Monthly Charges by Churn Status (Box Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x='Churn Label', y='Monthly Charges', hue='Churn Label',
            palette={'No': '#2A9D8F', 'Yes': '#E63946'}, ax=ax, legend=False)
ax.set_xlabel('Churn Status')
ax.set_ylabel('Monthly Charges ($)')
ax.set_title('Monthly Charges Distribution by Churn Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.5 Tenure vs. Total Charges (Scatter Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = df['Churn Label'].map({'No': '#2A9D8F', 'Yes': '#E63946'})
ax.scatter(df['Tenure Months'], df['Total Charges'], c=colors, alpha=0.4, s=15)
ax.set_xlabel('Tenure (Months)')
ax.set_ylabel('Total Charges ($)')
ax.set_title('Tenure vs. Total Charges, Colored by Churn Status', fontsize=14, fontweight='bold')
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=8, label=l)
           for l, c in [('No Churn', '#2A9D8F'), ('Churn', '#E63946')]]
ax.legend(handles=handles, title='Churn Status')
plt.tight_layout()
plt.show()

### 4.6 Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax, cbar_kws={'label': 'Correlation Coefficient'})
ax.set_title('Correlation Heatmap of Numeric Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.7 Monthly Revenue Trend by Tenure Group (Line Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
order = ['0-1 Year', '1-2 Years', '2-4 Years', '4-5 Years', '5+ Years']
trend = df.groupby('Tenure Group')['Avg Monthly Spend'].mean().reindex(order)
ax.plot(trend.index, trend.values, marker='o', linewidth=2.5, color='#2E86AB', markersize=8)
ax.set_xlabel('Tenure Group')
ax.set_ylabel('Average Monthly Spend ($)')
ax.set_title('Average Monthly Spend Across Tenure Groups', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.8 Top Churn Reasons (Horizontal Bar Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
reason_counts.sort_values().plot(kind='barh', ax=ax, color='#F4A261', edgecolor='black')
ax.set_xlabel('Number of Customers')
ax.set_ylabel('Churn Reason')
ax.set_title('Top 10 Reasons Customers Cited for Churning', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.9 Multi-Panel Overview (Subplots)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Churn rate by Internet Service
internet_churn = df.groupby('Internet Service')['Churn Value'].mean().mul(100)
axes[0, 0].bar(internet_churn.index, internet_churn.values, color=COLOR_PALETTE[:3], edgecolor='black')
axes[0, 0].set_title('Churn Rate by Internet Service')
axes[0, 0].set_ylabel('Churn Rate (%)')

# Panel 2: Churn rate by Payment Method
payment_churn = df.groupby('Payment Method')['Churn Value'].mean().mul(100).sort_values()
axes[0, 1].barh(payment_churn.index, payment_churn.values, color=COLOR_PALETTE[1], edgecolor='black')
axes[0, 1].set_title('Churn Rate by Payment Method')
axes[0, 1].set_xlabel('Churn Rate (%)')

# Panel 3: Services Subscribed distribution
sns.countplot(data=df, x='Services Subscribed', hue='Churn Label',
              palette={'No': '#2A9D8F', 'Yes': '#E63946'}, ax=axes[1, 0])
axes[1, 0].set_title('Number of Add-On Services vs. Churn')
axes[1, 0].set_xlabel('Services Subscribed (count)')

# Panel 4: CLTV Segment vs churn
sns.countplot(data=df, x='CLTV Segment', hue='Churn Label', order=['Low', 'Medium', 'High'],
              palette={'No': '#2A9D8F', 'Yes': '#E63946'}, ax=axes[1, 1])
axes[1, 1].set_title('CLTV Segment vs. Churn')
axes[1, 1].set_xlabel('CLTV Segment')

fig.suptitle('Churn Drivers — Multi-Panel Overview', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.10 Interactive Visualization — CLTV vs. Monthly Charges (Plotly)

In [ ]:
import plotly.express as px

fig = px.scatter(
    df, x='Monthly Charges', y='CLTV', color='Churn Label',
    color_discrete_map={'No': '#2A9D8F', 'Yes': '#E63946'},
    hover_data=['Tenure Months', 'Contract'],
    title='Customer Lifetime Value vs. Monthly Charges by Churn Status',
    labels={'Monthly Charges': 'Monthly Charges ($)', 'CLTV': 'Customer Lifetime Value'},
    opacity=0.6
)
fig.update_layout(legend_title_text='Churn Status', title_x=0.5)
fig.show()

## 5. Insight Generation and Summary

**Key Findings**

1. **Overall churn rate is ~26.5%** — roughly 1 in 4 customers left the company, which is high for a subscription-based business and signals a retention problem worth addressing.

2. **Contract type is the strongest churn driver.** Month-to-month customers churn at a dramatically higher rate than One-year or Two-year contract holders. Longer commitments correlate strongly with loyalty, likely because monthly customers face no switching penalty.

3. **Fiber optic internet customers churn more than DSL customers**, despite fiber typically being the premium service — suggesting dissatisfaction with fiber pricing, reliability, or customer support experience rather than the technology itself.

4. **Tenure and churn are inversely related.** Customers in their first year are far more likely to churn than long-tenured customers — the first 12 months represent the highest-risk retention window.

5. **Electronic check users churn more than customers on automatic payment methods** (bank transfer/credit card), hinting that friction or lack of commitment in the payment process may correlate with disengagement.

6. **Customers with fewer add-on services (Online Security, Tech Support, etc.) churn more.** Add-on subscriptions appear to increase "stickiness" — customers with more integrated services are more invested in staying.

7. **Top self-reported churn reasons** center on competitor offers (better prices, more data, higher speeds) and service attitude/support issues — pointing to both **competitive pricing pressure** and **customer service quality** as levers management can act on.

8. **Monthly Charges and Total Charges are moderately-to-strongly correlated with Churn Score**, reinforcing that higher-paying customers on short commitments are a particularly volatile segment.

**Recommendations / Next Steps**

- Prioritize retention offers (discounts, contract incentives) for **month-to-month, fiber-optic customers within their first 12 months** — this is the highest-risk segment.
- Investigate fiber optic service quality/pricing specifically, since it churns more than the technically simpler DSL service.
- Encourage adoption of add-on services (security, tech support) as a retention lever, potentially bundled at a discount for new customers.
- Review the electronic check payment experience and promote automatic payment enrollment.
- Use the `Churn Score` and `CLTV Segment` fields to build a prioritized, high-value "at-risk" customer list for proactive outreach by the retention team.

**Limitations**

- This is a cross-sectional snapshot rather than time-series data, so we can observe correlations but not prove causation.
- `Churn Reason` reflects self-reported, subjective feedback and may not capture the complete underlying cause of churn.
